<a href="https://colab.research.google.com/github/Binkai/OSCAL-Grundschutz-Analyse/blob/main/GS%2B%2BVergleich.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Skript 0: Split Grundschutz++

import os
import json
import re

# --- KONFIGURATION (Generische Pfade) ---
INPUT_FILE = "./daten/Grundschutz++-catalog.json"
OUTPUT_DIR = "./daten/json_praktiken"

def split_oscal_json():
    print(f"Lade JSON Datei: {INPUT_FILE}")

    # Ordner erstellen, falls nicht vorhanden
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # SCHRITT 1: JSON-Datei mit explizitem UTF-8 einlesen
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Fehler: Die Datei {INPUT_FILE} wurde nicht gefunden.")
        print("Bitte erstelle den Ordner './daten' und lege die Datei dort ab.")
        return
    except json.JSONDecodeError as e:
        print(f"❌ Fehler beim Parsen der JSON: {e}")
        return

    # Prüfen, ob die Struktur wie erwartet ist
    if 'catalog' not in data or 'groups' not in data['catalog']:
        print("❌ Fehler: Erwartete Struktur 'catalog' -> 'groups' nicht gefunden.")
        return

    top_level_groups = data['catalog']['groups']

    # SCHRITT 2 & 3: Extrahieren und Speichern
    for group in top_level_groups:
        # ID bereinigen (alles was kein Buchstabe/Zahl ist, wird zum Unterstrich)
        group_id = group.get('id', 'unbekannte_gruppe')
        clean_id = re.sub(r'[^a-zA-Z0-9]', '_', group_id)

        file_name = f"{clean_id}.json"
        file_path = os.path.join(OUTPUT_DIR, file_name)

        # Einzelne Gruppe als UTF-8 JSON speichern
        with open(file_path, 'w', encoding='utf-8') as f_out:
            # ensure_ascii=False ist extrem wichtig für deutsche Umlaute!
            json.dump(group, f_out, ensure_ascii=False, indent=4)

        print(f"✅ Erfolgreich gespeichert: {file_name}")

    print("\n🎉 Alle Dateien wurden mit korrekter Kodierung erstellt.")

# Aufruf der Funktion
split_oscal_json()

Lade JSON Datei: ./daten/Grundschutz++-catalog.json
✅ Erfolgreich gespeichert: GC.json
✅ Erfolgreich gespeichert: STM.json
✅ Erfolgreich gespeichert: UMS.json
✅ Erfolgreich gespeichert: VRB.json
✅ Erfolgreich gespeichert: PERF.json
✅ Erfolgreich gespeichert: ASST.json
✅ Erfolgreich gespeichert: PERS.json
✅ Erfolgreich gespeichert: BES.json
✅ Erfolgreich gespeichert: DLS.json
✅ Erfolgreich gespeichert: TEST.json
✅ Erfolgreich gespeichert: GEB.json
✅ Erfolgreich gespeichert: SENS.json
✅ Erfolgreich gespeichert: ARCH.json
✅ Erfolgreich gespeichert: BER.json
✅ Erfolgreich gespeichert: NOT.json
✅ Erfolgreich gespeichert: DET.json
✅ Erfolgreich gespeichert: REA.json
✅ Erfolgreich gespeichert: KONF.json
✅ Erfolgreich gespeichert: DEV.json

🎉 Alle Dateien wurden mit korrekter Kodierung erstellt.


In [ ]:
# @title Skript 0: Auswahl Gemini Modelle
from google import genai
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)

print("Verfügbare Modelle für 'generateContent':")
print("-" * 40)
for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(f"Name: {model.name}")
        print(f"Display Name: {model.display_name}")
        print("-" * 40)

In [ ]:
# @title Skript 1: Praktiken-Zusammenfassung (CSV -> MD)
import os
import glob
import time
from google import genai
from google.genai import types
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)
# Model-ID ggf. anpassen
MODEL_ID = "gemini-pro-latest"

# Pfade
INPUT_FOLDER = "./daten/json_praktiken"
OUTPUT_FOLDER =  "./daten/wissensbasis/praktiken_summary_PRO"

# Ordner erstellen
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# --- PROMPT ---
def create_summary_prompt(json_content, filename):
    return f"""
    Rolle: Du bist ein spezialisierter IT-Sicherheitsanalyst mit tiefer Expertise im BSI IT-Grundschutz und dem neuen Grundschutz++ Framework.
Aufgabe: Analysiere die nachfolgend bereitgestellte JSON-Datei, die eine Hauptgruppe (hier die Praktik {filename}) des Grundschutz++ im OSCAL-Format enthält. Erstelle eine strukturierte Zusammenfassung, die als Basis für einen späteren Vergleich mit dem IT-Grundschutz-Kompendium 2023 dient.
Erwartete Struktur der Antwort:
Stammdaten: ID und Titel der Hauptgruppe sowie die Kurzbeschreibung (aus den props).
Strategisches Ziel: Erkläre in 3–4 Sätzen: Welches Sicherheitsziel verfolgt diese Praktik? Welche organisatorische Fähigkeit (Capability) wird hier aufgebaut?
Struktur der Untergruppen: Liste die enthaltenen Untergruppen (z. B. GC.1, GC.2) mit ihren Titeln auf.
Anforderungs-Katalog (Tabellarisch): Erstelle eine Tabelle aller enthaltenen Einzelanforderungen (controls) mit folgenden Spalten:
ID: Die genaue Kennung (z. B. GC.1.1).
Titel: Bezeichnung der Anforderung.
Klasse: (z. B. normal-SdT, erhoeht).
Kern-Aussage: Eine Zusammenfassung des Inhalts oder eine Kopie des Wortlauts der Anforderung, wenn diese deiner Einschätzung nach kurz ist.
Besonderheiten: Notiere hier spezifische Parameter (params) oder neue Konzepte, die im Vergleich zum klassischen Grundschutz auffallen (z. B. Fokus auf Reifegrade oder Automatisierung).
Wichtig: Bleibe objektiv und präzise am Text der JSON-Datei. Vermeide allgemeine Floskeln.

    JSON Content:
    {json_content}
    """
def call_api_with_retry(contents, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
            )
            return response
        except Exception as e:
            error_str = str(e)
            if "503" in error_str or "429" in error_str:
                wait_time = (attempt + 1) * 15
                print(f"      ⚠️ Server Error (503). Warte {wait_time}s... (Versuch {attempt+1}/{retries})")
                time.sleep(wait_time)
            else:
                raise e
    return None
# --- MAIN LOOP ---
def main():
    print("Starte Zusammenfassung der Praktiken...")
    json_files = glob.glob(os.path.join(INPUT_FOLDER, "*.json"))

    if not json_files:
        print(f"Keine Dateien in {INPUT_FOLDER} gefunden.")
        return

    for i, json_file in enumerate(json_files):
        filename = os.path.basename(json_file)
        praktik_id = os.path.splitext(filename)[0]
        output_file = os.path.join(OUTPUT_FOLDER, f"{praktik_id}_Summary.md")

        print(f"[{i+1}/{len(json_files)}] Verarbeite {praktik_id}...")

        # Prüfen ob Datei schon existiert (Resume-Funktion)
        if os.path.exists(output_file):
            print(f"Datei existiert bereits. Überspringe.")
            continue

        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                content = f.read()

            response = call_api_with_retry(
                        contents=create_summary_prompt(content, praktik_id)
                    )

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(response.text)

            print(f" Gespeichert.")
            time.sleep(2) # Pause für Rate Limit

        except Exception as e:
            print(f"Fehler: {e}")

    print(f"\n Fertig! Dateien liegen in: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()

In [ ]:
# @title Skript 2: Baustein-Zusammenfassung (CSV -> MD)
import pandas as pd
import os
import time
from google import genai
from google.colab import userdata

# --- KONFIGURATION ---
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-3-flash-preview"

# Pfade
CSV_PATH = "/content/grundschutz_2023.csv"
OUTPUT_FOLDER = "/content/wissensbasis/bausteine_summary/"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# --- HELFER ---
def load_csv_safe(filepath):
    """Lädt CSV robust."""
    try:
        df = pd.read_csv(filepath, sep=';', encoding='utf-8')
    except:
        df = pd.read_csv(filepath, sep=',', encoding='latin-1')
    df.columns = [c.strip() for c in df.columns]
    return df

def get_baustein_id(identifier):
    """
    Macht aus 'SYS.1.1.A1' -> 'SYS.1.1'
    Logik: Alles vor dem letzten Punkt entfernen, wenn danach 'A' kommt.
    """
    parts = identifier.split('.')
    # Wenn das letzte Teil 'A' + Zahl ist (z.B. A1, A12), schneiden wir es ab
    if len(parts) > 1 and parts[-1].startswith('A') and parts[-1][1:].isdigit():
        return ".".join(parts[:-1])
    return identifier # Fallback

def create_baustein_prompt(baustein_id, requirements_text):
    return f"""
    Du bist ein IT-Grundschutz-Experte.
    Hier sind ALLE Einzelanforderungen des Bausteins '{baustein_id}' aus dem Kompendium 2023.

    Deine Aufgabe:
    Erstelle eine prägnante Zusammenfassung dieses Bausteins im Markdown-Format.
    Diese Zusammenfassung soll später genutzt werden, um zu prüfen, ob ein neuer Standard diese Themen abdeckt.

    Struktur:
    1. **Titel & Fokus**: Worum geht es in diesem Baustein primär?
    2. **Kerninhalte**: Fasse die Anforderungen zusammen. Welche Sicherheitsziele werden verfolgt? (Nicht jede Anforderung einzeln auflisten, sondern cluster die Themen, z.B. "Zutrittskontrolle", "Protokollierung", "Verschlüsselung").
    3. **Wichtige Details**: Gibt es spezifische Grenzwerte oder kritische Muss-Anforderungen, die hervorstechen?

    Liste der Anforderungen:
    {requirements_text}
    """

# --- MAIN LOOP ---
def main():
    print("🚀 Starte Analyse der CSV-Bausteine...")

    df = load_csv_safe(CSV_PATH)

    # Spalten checken
    req_cols = ['Identifier__(Baustein-Anforderung)', 'Anforderungstext__(Baustein-Anforderung)', 'Titel__(Baustein-Anforderung)']
    if not all(col in df.columns for col in req_cols):
        print("❌ Spaltennamen stimmen nicht. Bitte CSV prüfen.")
        return

    # Baustein-ID extrahieren und gruppieren
    df['Baustein_ID'] = df['Identifier__(Baustein-Anforderung)'].apply(get_baustein_id)
    grouped = df.groupby('Baustein_ID')

    total_bausteine = len(grouped)
    print(f"📋 {total_bausteine} einzigartige Bausteine identifiziert.")

    count = 0
    for baustein_id, group in grouped:
        count += 1
        output_file = os.path.join(OUTPUT_FOLDER, f"{baustein_id}_Summary.md")

        # Resume-Check
        if os.path.exists(output_file):
            print(f"[{count}/{total_bausteine}] {baustein_id} existiert schon. Skip.")
            continue

        print(f"[{count}/{total_bausteine}] Generiere Zusammenfassung für: {baustein_id} ({len(group)} Anforderungen)")

        # Textblock bauen
        full_text = ""
        for _, row in group.iterrows():
            full_text += f"ID: {row['Identifier__(Baustein-Anforderung)']}\n"
            full_text += f"Titel: {row['Titel__(Baustein-Anforderung)']}\n"
            full_text += f"Text: {row['Anforderungstext__(Baustein-Anforderung)']}\n\n"

        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=create_baustein_prompt(baustein_id, full_text)
            )

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(response.text)

            # WICHTIG: Rate Limit Pause. Da CSV viele Bausteine hat, hier etwas vorsichtiger sein.
            time.sleep(3.5)

        except Exception as e:
            print(f"   ❌ Fehler bei {baustein_id}: {e}")
            time.sleep(10) # Längere Pause bei Fehler

    print(f"\n🎉 Fertig! Baustein-Zusammenfassungen in: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    main()

In [ ]:
# Ordner als ZIP packen und herunterladen
!zip -r /content/ergebnisse.zip /content/wissensbasis/praktiken_summary/
from google.colab import files
files.download("/content/ergebnisse.zip")

In [ ]:
# @title Skript 3: Intelligentes Mapping (Mit Kurzzusammenfassungen)
!pip install -q -U google-genai pandas

import os
import glob
import json
import pandas as pd
import time
from google import genai
from google.genai import types
from google.colab import userdata

# --- KONFIGURATION ---
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-pro-latest"

# Pfade
PRAKTIKEN_MD_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/wissensbasis/praktiken_summary_PRO"
# Hier den Namen deiner neuen Datei eintragen:
MAPPING_TABLE_PATH = "/content/drive/MyDrive/Projekt_Grundschutz++/mapping_Referenz.csv"
OUTPUT_MAPPING_FILE = "/content/drive/MyDrive/Projekt_Grundschutz++/content/wissensbasis/final_mapping_PRO.json"

# --- HELFER: INDEX AUS DEINER TABELLE BAUEN ---
def create_enhanced_index(csv_path):
    """
    Liest die vorbereitete Tabelle mit Schicht und Zusammenfassung ein.
    Erwartete Spalten (ungefähr): 'Schicht', 'Baustein-ID', 'Titel', 'Kurzzusammenfassung des Inhalts'
    """
    print(f"Lese Mapping-Tabelle: {csv_path}")

    # Robustes Laden (Semikolon oder Komma)
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    except:
        try:
            df = pd.read_csv(csv_path, sep=',', encoding='latin-1') # Fallback für Excel-CSVs
        except Exception as e:
            print(f"Fehler beim Laden der CSV: {e}")
            return ""

    # Spalten bereinigen (Leerzeichen entfernen)
    df.columns = [c.strip() for c in df.columns]

    # Mapping der Spaltennamen (Falls sie in der CSV leicht anders heißen)
    # Wir suchen die passendste Spalte für jeden Datentyp
    col_map = {}
    for col in df.columns:
        if "schicht" in col.lower(): col_map['schicht'] = col
        elif "id" in col.lower() and "baustein" in col.lower(): col_map['id'] = col
        elif "titel" in col.lower(): col_map['titel'] = col
        elif "zusammenfassung" in col.lower() or "inhalt" in col.lower(): col_map['summary'] = col

    # Check ob wir alles gefunden haben
    if len(col_map) < 4:
        print(f"Warnung: Konnte nicht alle Spalten automatisch identifizieren. Gefunden: {col_map}")
        print(f"Verfügbare Spalten in CSV: {list(df.columns)}")
        # Wir machen trotzdem weiter, falls kritische Spalten da sind

    context_text = "LISTE DER VERFÜGBAREN ALTEN BAUSTEINE (Referenz 2023):\n"
    context_text += "Format: [Schicht] ID - Titel: Inhalt\n\n"

    for _, row in df.iterrows():
        # Werte holen (mit Fallback, falls Spalte fehlt)
        schicht = row.get(col_map.get('schicht', 'N/A'), '')
        b_id = row.get(col_map.get('id', 'N/A'), '')
        titel = row.get(col_map.get('titel', 'N/A'), '')
        summary = row.get(col_map.get('summary', 'N/A'), '')

        # Zeile für den Prompt bauen
        line = f"[{schicht}] {b_id} - {titel}: {summary}"
        context_text += line + "\n"

    return context_text

# --- PROMPT ---
def create_mapping_prompt(praktik_summary, baustein_index):
    return f"""
    Du bist ein Experte für BSI IT-Grundschutz Migrationen.

    INPUT 1: NEUE PRAKTIK (Grundschutz++ Zusammenfassung):
    {praktik_summary}

    INPUT 2: ALTE BAUSTEINE (Referenz-Liste mit Inhaltsangabe):
    {baustein_index}

    AUFGABE:
    Analysiere Input 1 (Praktik). Identifiziere aus Input 2 (Baustein-Liste), welche alten Bausteine inhaltlich in dieser neuen Praktik aufgehen oder thematisch stark verwandt sind.

    REGELN:
    1. Nutze die Inhaltsbeschreibungen für eine präzise Zuordnung.
    2. Nenne nur Bausteine, die wirklich relevant sind (High Confidence).
    3. Gib das Ergebnis AUSSCHLIESSLICH als JSON-Liste von Strings (nur die IDs) zurück.

    Beispiel Output:
    ["CON.1", "SYS.1.2"]
    """

# --- MAIN ---
def main():
    # 1. Index erstellen
    baustein_index_str = create_enhanced_index(MAPPING_TABLE_PATH)
    if not baustein_index_str: return

    print(f"   Index erstellt. Größe: {len(baustein_index_str)} Zeichen.")

    md_files = glob.glob(os.path.join(PRAKTIKEN_MD_DIR, "*.md"))
    full_mapping = {}

    print(f"\n Starte Mapping für {len(md_files)} Praktiken...")

    for md_file in md_files:
        filename = os.path.basename(md_file)
        # Dateiname ist z.B. "OPS_Summary.md" -> ID ist "OPS"
        # Wir entfernen "_Summary.md" um die reine ID zu bekommen
        praktik_id = filename.replace("_Summary.md", "")

        print(f"   Mappe Praktik: {praktik_id}")

        with open(md_file, 'r', encoding='utf-8') as f:
            summary_content = f.read()

        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=create_mapping_prompt(summary_content, baustein_index_str),
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                )
            )

            mapping_list = json.loads(response.text)
            full_mapping[praktik_id] = mapping_list
            print(f"      Treffer: {mapping_list}")

            time.sleep(1)

        except Exception as e:
            print(f"      Fehler: {e}")
            full_mapping[praktik_id] = []

    # Speichern
    with open(OUTPUT_MAPPING_FILE, 'w', encoding='utf-8') as f:
        json.dump(full_mapping, f, indent=4, ensure_ascii=False)

    print(f"\nMapping abgeschlossen. Gespeichert in: {OUTPUT_MAPPING_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
# @title Skript 4: Finaler Gap-Vergleich
!pip install -q -U google-genai pandas

import os
import glob
import json
import pandas as pd
import time
from google import genai
from google.genai import types
from google.colab import userdata

# --- KONFIGURATION ---
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-pro-latest"

# Pfade
PRAKTIKEN_MD_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/wissensbasis/praktiken_summary_PRO"
SPLIT_CSV_DIR = "/content/bausteine_csv_split/"
MAPPING_FILE = "/content/drive/MyDrive/Projekt_Grundschutz++/content/wissensbasis/final_mapping_PRO.json"
OUTPUT_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/ergebnisse/Vergleich_PRO"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- HELFER ---

def find_csv_for_baustein(baustein_id, search_dir):
    """
    Sucht die passende CSV-Datei für eine Baustein-ID.
    Da die Dateinamen oft 'CON.1_Kryptokonzept.csv' heißen, suchen wir nach dem Prefix.
    """
    # Alle CSVs im Ordner
    files = glob.glob(os.path.join(search_dir, "*.csv"))

    # Exakter Match oder Starts-With (mit Punkt oder Unterstrich danach, um CON.1 nicht mit CON.10 zu verwechseln)
    for f in files:
        fname = os.path.basename(f)
        if fname.startswith(f"{baustein_id}.csv") or \
           fname.startswith(f"{baustein_id}_") or \
           fname.startswith(f"{baustein_id} "):
            return f
    return None

def load_requirements_text(csv_path):
    """Liest die CSV und baut einen lesbaren Textblock aller Anforderungen."""
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    except:
        try:
            df = pd.read_csv(csv_path, sep=',', encoding='latin-1')
        except:
            return None

    # Spalten finden
    cols = [c.strip() for c in df.columns]
    df.columns = cols

    # Wir suchen Spalten für ID und Text
    id_col = next((c for c in cols if "Ident" in c), None)
    text_col = next((c for c in cols if "Anforderungstext" in c or "Text" in c), None)
    title_col = next((c for c in cols if "Titel" in c), None)

    if not id_col or not text_col:
        return "FEHLER: Spalten in CSV nicht identifizierbar."

    text_block = ""
    for _, row in df.iterrows():
        # Format: [ID] Titel: Text
        text_block += f"[{row[id_col]}] {row.get(title_col, '')}\n"
        text_block += f"Inhalt: {row[text_col]}\n\n"

    return text_block

def create_comparison_prompt(praktik_name, praktik_summary, baustein_id, csv_requirements):
    return f"""
    Kontext: Ich führe eine Gap-Analyse zwischen dem IT-Grundschutz 2023 und dem neuen Grundschutz++ durch.


    Input 1: Der alte, bisherige IT-Grundschutz 2023:
    Baustein: {baustein_id}
    Anforderungen (aus CSV):
    {csv_requirements}

    Input 2:Hier ist die Analyse der JSON-Datei für die Praktik {praktik_name}:
    {praktik_summary}

    AUFGABE:
    Vergleiche den alten Baustein '{baustein_id}' mit der neuen Praktik {praktik_name}.
    Erstelle eine Markdown-Tabelle:
    | {praktik_name}-ID | '{baustein_id}'-ID | Status im GS++ | Anmerkung/Mapping |

    Status: "Identisch", "Teilweise identisch", "Neu", "Verschoben".
    Hebe hervor, wenn Anforderungen strenger oder unpräziser geworden sind.
    Liste zusätzlich auf:
    - Fehlende Anforderungen im neuen Modell.
    """
def call_api_with_retry(contents, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
            )
            return response
        except Exception as e:
            error_str = str(e)
            if "503" in error_str or "429" in error_str:
                wait_time = (attempt + 1) * 15
                print(f" Server Error (503). Warte {wait_time}s... (Versuch {attempt+1}/{retries})")
                time.sleep(wait_time)
            else:
                raise e
    return None
# --- MAIN ---

def main():
    print("Starte finalen Gap-Vergleich...")

    # 1. Mapping laden
    if not os.path.exists(MAPPING_FILE):
        print(" Mapping-Datei fehlt.")
        return

    with open(MAPPING_FILE, 'r', encoding='utf-8') as f:
        mapping_data = json.load(f)

    # 2. Iteration über Praktiken
    for praktik_id, baustein_list in mapping_data.items():
        if not baustein_list:
            print(f"{praktik_id}: Keine zugeordneten Bausteine (leer).")
            continue

        print(f"\nAnalysiere Praktik: {praktik_id}")
        print(f"   Zugeordnete Bausteine: {baustein_list}")

        # Summary laden
        summary_path = os.path.join(PRAKTIKEN_MD_DIR, f"{praktik_id}_Summary.md")
        if not os.path.exists(summary_path):
            print(f"Keine Summary gefunden für {praktik_id}. Überspringe.")
            continue

        with open(summary_path, 'r', encoding='utf-8') as f:
            praktik_summary = f.read()

        # Output Datei für diese Praktik
        output_file = os.path.join(OUTPUT_DIR, f"Gap_Analyse_{praktik_id}.md")

        with open(output_file, 'w', encoding='utf-8') as f_out:
            f_out.write(f"# Gap-Analyse Report: {praktik_id}\n\n")
            f_out.write(f"Basierend auf Mapping: {baustein_list}\n\n")

            # 3. Iteration über zugeordnete Bausteine
            for b_id in baustein_list:
                print(f"  Vergleiche mit Baustein: {b_id} ...")

                # CSV suchen
                csv_path = find_csv_for_baustein(b_id, SPLIT_CSV_DIR)
                if not csv_path:
                    print(f"CSV für {b_id} nicht gefunden!")
                    f_out.write(f"## Fehler: Baustein {b_id} nicht als CSV verfügbar.\n\n")
                    continue

                # Anforderungen laden
                req_text = load_requirements_text(csv_path)
                if not req_text:
                    print(f"Konnte Anforderungen aus {csv_path} nicht lesen.")
                    continue
                # API Call
                try:
                    response = call_api_with_retry(
                        contents=create_comparison_prompt(praktik_id, praktik_summary, b_id, req_text)
                    )

                    f_out.write(response.text)
                    f_out.write("\n\n---\n\n")
                    print("      ✅ Analyse erstellt.")
                    time.sleep(3) # Rate Limit Pause

                except Exception as e:
                    print(f"API Fehler: {e}")
                    f_out.write(f"API Fehler bei {b_id}: {e}\n\n")

    print(f"\nAlle Vergleiche abgeschlossen. Ergebnisse in: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

In [ ]:
# @title Skript 5: Repair & Retry (Nur fehlende Bausteine)
!pip install -q -U google-genai pandas

import os
import glob
import json
import pandas as pd
import time
from google import genai
from google.genai import types
from google.colab import userdata

# --- KONFIGURATION ---
API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-flash-latest"


# Pfade (Exakt wie vorher)
PRAKTIKEN_MD_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/wissensbasis/praktiken_summary"
SPLIT_CSV_DIR = "/content/bausteine_csv_split/"
MAPPING_FILE = "/content/drive/MyDrive/Projekt_Grundschutz++/content/final_mapping.json"
OUTPUT_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/ergebnisse/Vergleich"



def create_comparison_prompt(praktik_name, praktik_summary, baustein_id, csv_requirements):
    return f"""
    Kontext: Ich führe eine Gap-Analyse zwischen dem IT-Grundschutz 2023 und dem neuen Grundschutz++ durch.


    Input 1: Der alte, bisherige IT-Grundschutz 2023:
    Baustein: {baustein_id}
    Anforderungen (aus CSV):
    {csv_requirements}

    Input 2:Hier ist die Analyse der JSON-Datei für die Praktik {praktik_name}:
    {praktik_summary}

    AUFGABE:
    Vergleiche den alten Baustein '{baustein_id}' mit der neuen Praktik {praktik_name}.
    Erstelle eine Markdown-Tabelle:
    | {praktik_name}-ID | '{baustein_id}'-ID | Status im GS++ | Anmerkung/Mapping |

    Status: "Identisch", "Teilweise identisch", "Neu", "Verschoben".
    Hebe hervor, wenn Anforderungen strenger oder unpräziser geworden sind.
    Liste zusätzlich auf:
    - Fehlende Anforderungen im neuen Modell.
    Beginne deine Antwort ZWINGEND mit folgenden Hinweis und bearbeite dann die Aufgabe: "Vergleich Baustein: {baustein_id} mit Praktik: {praktik_name}."
    """
# --- HELFER ---

def find_csv_for_baustein(baustein_id, search_dir):
    files = glob.glob(os.path.join(search_dir, "*.csv"))
    for f in files:
        fname = os.path.basename(f)
        if fname.startswith(f"{baustein_id}.csv") or \
           fname.startswith(f"{baustein_id}_") or \
           fname.startswith(f"{baustein_id} "):
            return f
    return None

def get_verification_marker(csv_path):
    """
    Holt die erste Anforderungs-ID aus der CSV (z.B. 'INF.1.A1').
    Das dient als 'Fingerabdruck', um zu prüfen, ob der Baustein schon bearbeitet wurde.
    """
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    except:
        try:
            df = pd.read_csv(csv_path, sep=',', encoding='latin-1')
        except:
            return None

    # Spalten bereinigen
    df.columns = [c.strip() for c in df.columns]

    # ID Spalte suchen
    id_col = next((c for c in df.columns if "Ident" in c), None)

    if id_col and not df.empty:
        # Nimm die erste ID (z.B. SIS.1.A1)
        first_id = str(df.iloc[0][id_col]).strip()
        return first_id
    return None

def load_requirements_text(csv_path):
    # (Gleiche Funktion wie vorher zum Laden des vollen Textes)
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='utf-8')
    except:
        try:
            df = pd.read_csv(csv_path, sep=',', encoding='latin-1')
        except:
            return None
    cols = [c.strip() for c in df.columns]
    df.columns = cols
    id_col = next((c for c in cols if "Ident" in c), None)
    text_col = next((c for c in cols if "Anforderungstext" in c or "Text" in c), None)
    title_col = next((c for c in cols if "Titel" in c), None)
    if not id_col or not text_col: return None
    text_block = ""
    for _, row in df.iterrows():
        text_block += f"[{row[id_col]}] {row.get(title_col, '')}\nInhalt: {row[text_col]}\n\n"
    return text_block

def call_api_with_retry(contents, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents
            )
            return response
        except Exception as e:
            error_str = str(e)
            if "503" in error_str or "429" in error_str:
                wait_time = (attempt + 1) * 15
                print(f"      ⚠️ Server Error (503). Warte {wait_time}s... (Versuch {attempt+1}/{retries})")
                time.sleep(wait_time)
            else:
                raise e
    return None

# --- MAIN ---

def main():
    print("🛠️ Starte ROBUSTEN Reparatur-Modus (Check auf Anforderungs-IDs)...")

    if not os.path.exists(MAPPING_FILE):
        print("❌ Mapping-Datei fehlt.")
        return

    with open(MAPPING_FILE, 'r', encoding='utf-8') as f:
        mapping_data = json.load(f)

    for praktik_id, baustein_list in mapping_data.items():
        if not baustein_list: continue

        output_file = os.path.join(OUTPUT_DIR, f"Gap_Analyse_{praktik_id}.md")

        # 1. Existing Content lesen
        existing_content = ""
        if os.path.exists(output_file):
            with open(output_file, 'r', encoding='utf-8') as f_read:
                existing_content = f_read.read()
        else:
            with open(output_file, 'w', encoding='utf-8') as f_new:
                f_new.write(f"# Gap-Analyse Report: {praktik_id}\n\n")

        praktik_summary = None

        for b_id in baustein_list:
            # CSV suchen
            csv_path = find_csv_for_baustein(b_id, SPLIT_CSV_DIR)
            if not csv_path:
                print(f"      ❌ CSV nicht gefunden: {b_id}")
                continue

            # --- DER NEUE CHECK ---
            # Wir holen die erste Anforderung (z.B. "CON.1.A1")
            verification_marker = get_verification_marker(csv_path)

            is_processed = False

            # Methode A: Suche nach Marker (Anforderungs-ID)
            if verification_marker and verification_marker in existing_content:
                is_processed = True

            # Methode B: Fallback - Suche nach Baustein-Header (falls Anforderung zufällig fehlt)
            # Wir suchen nach "Analyse: CON.1 " (mit Leerzeichen danach, um CON.12 auszuschließen)
            elif f"Analyse: {b_id}" in existing_content or f"Analyse: {b_id}\n" in existing_content:
                is_processed = True

            if is_processed:
                print(f"   ✅ {praktik_id} -> {b_id}: Vorhanden (Marker: {verification_marker}). Skip.")
                continue

            # ----------------------

            print(f"   🔧 {praktik_id} -> {b_id}: FEHLT. Lade Daten...")

            if not praktik_summary:
                sum_path = os.path.join(PRAKTIKEN_MD_DIR, f"{praktik_id}_Summary.md")
                if os.path.exists(sum_path):
                    with open(sum_path, 'r', encoding='utf-8') as f: praktik_summary = f.read()
                else:
                    break

            req_text = load_requirements_text(csv_path)
            if not req_text: continue

            try:
                response = call_api_with_retry(
                    create_comparison_prompt(praktik_id, praktik_summary, b_id, req_text)
                )

                if response and response.text:
                    with open(output_file, 'a', encoding='utf-8') as f_out:
                        f_out.write(f"\n{response.text}\n\n---\n\n")
                    print("      ✅ Nachgeholt und gespeichert.")
                else:
                    print("      ❌ Fehlgeschlagen (Kein Text).")

            except Exception as e:
                print(f"      ❌ Fehler: {e}")

    print("\n🎉 Reparatur abgeschlossen.")

if __name__ == "__main__":
    main()

In [ ]:
# @title Skript: Bereinigung von API-Fehlermeldungen
import os
import glob

# --- KONFIGURATION ---
# Pfad zu deinen Vergleichsdateien
INPUT_DIR = "/content/drive/MyDrive/Projekt_Grundschutz++/content/ergebnisse/Vergleich_PRO"

# Textmuster, nach dem gesucht werden soll (Teilstring reicht)
ERROR_PATTERN = "API Fehler bei"

# --- MAIN ---
def main():
    print(f"Starte Bereinigung im Ordner: {INPUT_DIR}")

    md_files = glob.glob(os.path.join(INPUT_DIR, "*.md"))

    if not md_files:
        print("Keine Markdown-Dateien gefunden.")
        return

    total_removed = 0
    files_affected = 0

    for file_path in md_files:
        filename = os.path.basename(file_path)

        # Datei einlesen
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        new_lines = []
        file_changed = False

        # Zeilen filtern
        for line in lines:
            if ERROR_PATTERN in line:
                # Wir haben eine Fehlerzeile gefunden -> NICHT übernehmen
                print(f"Entferne Fehlerzeile in {filename}: {line.strip()[:50]}...")
                file_changed = True
                total_removed += 1
            else:
                # Zeile ist okay -> übernehmen
                new_lines.append(line)

        # Datei überschreiben, nur wenn Änderungen vorliegen
        if file_changed:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.writelines(new_lines)
            files_affected += 1

    print("\n" + "="*40)
    print(f"Bereinigung abgeschlossen.")
    print(f"Betroffene Dateien: {files_affected}")
    print(f"Entfernte Fehlerzeilen: {total_removed}")
    print("="*40)

if __name__ == "__main__":
    main()

In [ ]:
# @title CSV Splitter (Keine API notwendig)
import pandas as pd
import os
import shutil

# --- KONFIGURATION ---
# Passe den Dateinamen an, falls er bei dir anders heißt
INPUT_CSV = "/content/drive/MyDrive/Projekt_Grundschutz++/Alle_Bausteine.csv"
OUTPUT_DIR = "/content/bausteine_csv_split/"

# Ordner bereinigen/erstellen
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- HILFSFUNKTIONEN ---

def load_csv_safe(filepath):
    """Lädt die CSV robust (Semicolon oder Comma, UTF-8 oder Latin-1)."""
    try:
        print("   Versuche UTF-8 mit Semikolon...")
        df = pd.read_csv(filepath, sep=';', encoding='utf-8')
    except Exception:
        try:
            print("   Versuche Latin-1 mit Komma...")
            df = pd.read_csv(filepath, sep=',', encoding='latin-1')
        except Exception as e:
            print("Kritischer Fehler beim Laden: {e}")
            return None

    # Spaltennamen bereinigen (Leerzeichen entfernen)
    df.columns = [c.strip() for c in df.columns]
    return df

def extract_baustein_id(identifier):
    """
    Extrahiert die Baustein-ID aus der Anforderungs-ID.
    Logik: SYS.1.1.A1 -> SYS.1.1
    """
    if pd.isna(identifier):
        return "UNKNOWN"

    parts = str(identifier).split('.')
    # Wenn das letzte Element wie eine Anforderungs-ID aussieht (z.B. A1, A12, M1)
    # schneiden wir es ab. Bausteine haben meistens Struktur XXX.Y.Z

    # Einfache Heuristik: Wenn mehr als 2 Teile und letzter Teil beginnt mit Buchstabe + Zahl
    if len(parts) > 1:
        last_part = parts[-1]
        # Prüfen ob letzter Teil Anforderungskürzel ist (A... oder M...)
        if (last_part.startswith('A') or last_part.startswith('M')) and len(last_part) > 1 and last_part[1].isdigit():
             return ".".join(parts[:-1])

        # Manchmal ist die ID auch nur der Baustein selbst (selten in Anforderungsliste)
        return ".".join(parts[:-1])

    return identifier

# --- MAIN ---

def main():
    print(f"Starte Split-Vorgang für: {INPUT_CSV}")

    # 1. Laden
    df = load_csv_safe(INPUT_CSV)
    if df is None: return

    print(f" CSV geladen. {len(df)} Zeilen gesamt.")

    # Spalten prüfen
    target_col = 'Identifier__(Baustein-Anforderung)'
    if target_col not in df.columns:
        print(f"Spalte '{target_col}' nicht gefunden. Verfügbar: {list(df.columns)}")
        # Fallback versuchen, falls Spalte anders heißt
        possible_cols = [c for c in df.columns if 'Ident' in c]
        if possible_cols:
            print(f"Meintest du vielleicht: {possible_cols[0]}? Nutze diese.")
            target_col = possible_cols[0]
        else:
            return

    # 2. Baustein-ID ableiten
    print("Berechne Baustein-Zuordnung...")
    df['Baustein_Group'] = df[target_col].apply(extract_baustein_id)

    # Nach Gruppen iterieren
    grouped = df.groupby('Baustein_Group')
    unique_bausteine = len(grouped)

    print(f"Erstelle {unique_bausteine} einzelne CSV-Dateien in {OUTPUT_DIR}...")

    count = 0
    for name, group in grouped:
        # Dateiname bereinigen (Slashes entfernen falls vorhanden)
        safe_name = str(name).replace('/', '_').replace('\\', '_')
        filename = os.path.join(OUTPUT_DIR, f"{safe_name}.csv")

        # Speichern (Semikolon getrennt, UTF-8, ohne Index)
        group.to_csv(filename, sep=';', encoding='utf-8', index=False)
        count += 1

        if count % 20 == 0:
            print(f"      ... {count} Dateien erstellt")

    print(f"\nAlle {count} Bausteine wurden exportiert.")

if __name__ == "__main__":
    main()

In [ ]:
# @title Ordner zippen und herunterladen
!zip -r /content/Summary.zip /content/praktiken_summary_PRO/
from google.colab import files
files.download("/content/Summary.zip")